In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

c:\Users\User\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
project_folder = Path.cwd().parent

uob_folder = project_folder / "Year_End_2026" / "UOB_Account"
output_folder = project_folder / "Datamart"

output_folder.mkdir(parents=True, exist_ok=True)

print("Current notebook folder:", Path.cwd())
print("UOB source folder:", uob_folder)
print("Output folder:", output_folder)
print("UOB folder exists:", uob_folder.exists())

Current notebook folder: c:\Users\User\Desktop\Python 100 Days\Bank_Account_Project\Python_Command
UOB source folder: c:\Users\User\Desktop\Python 100 Days\Bank_Account_Project\Year_End_2026\UOB_Account
Output folder: c:\Users\User\Desktop\Python 100 Days\Bank_Account_Project\Datamart
UOB folder exists: True


In [3]:
uob_files = sorted(uob_folder.rglob("*.csv"))

print(f"Total CSV files found: {len(uob_files)}")

for file in uob_files:
    print(file.relative_to(uob_folder))

Total CSV files found: 5
Apr\UOB.csv
Feb\statment UOB-ก.พ.69.csv
Jan\UOB-6041259598-ม.ค.69.csv
Mar\UOB.csv
May\UOB-พ.ค.69.csv


In [4]:
sample_file = uob_files[0]

print("Sample file:", sample_file)

sample_raw = pd.read_csv(
    sample_file,
    encoding="utf-8-sig",
    sep=",",
    engine="python",
    skiprows=3,
    on_bad_lines="skip"
)

display(sample_raw.head())

Sample file: c:\Users\User\Desktop\Python 100 Days\Bank_Account_Project\Year_End_2026\UOB_Account\Apr\UOB.csv


,D2,6041259598,02/04/2026,02/04/2026 .1,07:00:29 AM,MISC CR,UOB CARD SETTLEMENT,UOB-000900800016024,,.1,MID 000900800016024,.2,.3,.4,.5,.6,.7,"25,130.08",0,"1,863,514.86"
0,D2,6041259598,03/04/2026,03/04/2026,07:00:28 AM,MISC CR,UOB CARD SETTLEMENT,UOB-000900800016081,,,MID 000900800016081,,,,,,,"57,753.00",0,"1,921,267.86"
1,D2,6041259598,03/04/2026,03/04/2026,07:00:28 AM,MISC CR,UOB CARD SETTLEMENT,UOB-000900800016099,,,MID 000900800016099,,,,,,,"33,929.89",0,"1,955,197.75"
2,D2,6041259598,07/04/2026,07/04/2026,07:00:28 AM,MISC CR,UOB CARD SETTLEMENT,UOB-000900800016008,,,MID 000900800016008,,,,,,,"30,262.57",0,"1,985,460.32"
3,D2,6041259598,07/04/2026,07/04/2026,07:00:29 AM,MISC CR,UOB CARD SETTLEMENT,UOB-000900800017568,,,MID 000900800017568,,,,,,,"19,679.00",0,"2,005,139.32"
4,D2,6041259598,08/04/2026,08/04/2026,03:03:53 AM,MISC DR ITMX,IHAVE,CMDC3 PAYMENT,,,,,,,,,,0,"2,000,000.00","5,139.32"


In [5]:
print("Number of columns:", len(sample_raw.columns))

for index, column in enumerate(sample_raw.columns):
    print(index, repr(column))

Number of columns: 20
0 ' D2 '
1 '6041259598'
2 ' 02/04/2026 '
3 ' 02/04/2026 .1'
4 ' 07:00:29 AM '
5 ' MISC CR '
6 ' UOB CARD SETTLEMENT '
7 ' UOB-000900800016024 '
8 '  '
9 '  .1'
10 ' MID 000900800016024 '
11 '  .2'
12 '  .3'
13 '  .4'
14 '  .5'
15 '  .6'
16 '  .7'
17 '25,130.08'
18 '0'
19 '1,863,514.86'


In [6]:
def clean_column_names(df):
    df = df.copy()

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
    )

    df = df.loc[
        :,
        ~df.columns.str.contains(
            r"^Unnamed",
            case=False,
            regex=True
        )
    ]

    return df

In [7]:
sample_clean_columns = clean_column_names(sample_raw)

for index, column in enumerate(sample_clean_columns.columns):
    print(index, repr(column))

0 'D2'
1 '6041259598'
2 '02/04/2026'
3 '02/04/2026 .1'
4 '07:00:29 AM'
5 'MISC CR'
6 'UOB CARD SETTLEMENT'
7 'UOB-000900800016024'
8 ''
9 '.1'
10 'MID 000900800016024'
11 '.2'
12 '.3'
13 '.4'
14 '.5'
15 '.6'
16 '.7'
17 '25,130.08'
18 '0'
19 '1,863,514.86'


In [8]:
file_column_check = []

for file in uob_files:
    try:
        df_check = pd.read_csv(
            file,
            encoding="utf-8-sig",
            sep=",",
            engine="python",
            skiprows=3,
            on_bad_lines="skip"
        )

        df_check = clean_column_names(df_check)

        file_column_check.append(
            {
                "month_folder": file.parent.name,
                "source_file": file.name,
                "row_count": len(df_check),
                "column_count": len(df_check.columns),
                "columns": " | ".join(df_check.columns)
            }
        )

    except Exception as error:
        file_column_check.append(
            {
                "month_folder": file.parent.name,
                "source_file": file.name,
                "row_count": np.nan,
                "column_count": np.nan,
                "columns": f"ERROR: {error}"
            }
        )

column_check_df = pd.DataFrame(file_column_check)

display(column_check_df)

,month_folder,source_file,row_count,column_count,columns
0,Apr,UOB.csv,43,20,D2 | 6041259598 | 02/04/2026 | 02/04/2026 .1 |...
1,Feb,statment UOB-ก.พ.69.csv,40,20,D1 | Account Number | Value Date | Date | Time...
2,Jan,UOB-6041259598-ม.ค.69.csv,29,20,D2 | 6041259598 | 05/01/2026 | 05/01/2026 .1 |...
3,Mar,UOB.csv,50,20,D1 | Account Number | Value Date | Date | Time...
4,May,UOB-พ.ค.69.csv,46,20,D1 | Account Number | Value Date | Date | Time...


In [9]:
required_columns = {"account_number", "value_date"}

file_column_check = []

for file in uob_files:
    try:
        df_check = pd.read_csv(
            file,
            encoding="utf-8-sig",
            sep=",",
            engine="python",
            skiprows=3,
            on_bad_lines="skip"
        )

        df_check = clean_column_names(df_check)

        # Normalize again only for checking
        normalized_columns = {
            str(column)
            .strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
            for column in df_check.columns
        }

        has_account_number = "account_number" in normalized_columns
        has_value_date = "value_date" in normalized_columns

        missing_columns = sorted(required_columns - normalized_columns)

        file_column_check.append(
            {
                "month_folder": file.parent.name,
                "source_file": file.name,
                "row_count": len(df_check),
                "column_count": len(df_check.columns),

                # New checking columns
                "has_account_number": has_account_number,
                "has_value_date": has_value_date,
                "required_columns_present": len(missing_columns) == 0,
                "missing_columns": " | ".join(missing_columns),

                "columns": " | ".join(map(str, df_check.columns))
            }
        )

    except Exception as error:
        file_column_check.append(
            {
                "month_folder": file.parent.name,
                "source_file": file.name,
                "row_count": np.nan,
                "column_count": np.nan,
                "has_account_number": False,
                "has_value_date": False,
                "required_columns_present": False,
                "missing_columns": "ERROR",
                "columns": f"ERROR: {error}"
            }
        )

column_check_df = pd.DataFrame(file_column_check)

display(column_check_df)

,month_folder,source_file,row_count,column_count,has_account_number,has_value_date,required_columns_present,missing_columns,columns
0,Apr,UOB.csv,43,20,False,False,False,account_number | value_date,D2 | 6041259598 | 02/04/2026 | 02/04/2026 .1 |...
1,Feb,statment UOB-ก.พ.69.csv,40,20,True,True,True,,D1 | Account Number | Value Date | Date | Time...
2,Jan,UOB-6041259598-ม.ค.69.csv,29,20,False,False,False,account_number | value_date,D2 | 6041259598 | 05/01/2026 | 05/01/2026 .1 |...
3,Mar,UOB.csv,50,20,True,True,True,,D1 | Account Number | Value Date | Date | Time...
4,May,UOB-พ.ค.69.csv,46,20,True,True,True,,D1 | Account Number | Value Date | Date | Time...


In [10]:
import csv
import re
import pandas as pd
import numpy as np


def normalize_header_text(value):
    value = "" if value is None else str(value)

    return (
        value
        .replace("\ufeff", "")
        .replace("\xa0", " ")
        .strip()
        .lower()
    )


def find_uob_header_row(file, max_rows=30):
    with open(
        file,
        mode="r",
        encoding="utf-8-sig",
        errors="replace",
        newline=""
    ) as csv_file:

        reader = csv.reader(csv_file)

        for row_index, row in enumerate(reader):
            if row_index >= max_rows:
                break

            # Combine the full row so minor column-format differences do not matter
            row_text = " | ".join(
                normalize_header_text(value)
                for value in row
            )

            # Convert repeated spaces to one space
            row_text = re.sub(r"\s+", " ", row_text)

            has_account_number = (
                "account number" in row_text
                or "account_number" in row_text
            )

            has_value_date = (
                "value date" in row_text
                or "value_date" in row_text
            )

            if has_account_number and has_value_date:
                return row_index

    return None

In [11]:
required_columns = {
    "account_number",
    "value_date"
}

file_column_check = []

for file in uob_files:
    try:
        header_row = find_uob_header_row(file)

        if header_row is None:
            raise ValueError(
                "Cannot find a row containing Account Number and Value Date"
            )

        df_check = pd.read_csv(
            file,
            encoding="utf-8-sig",
            sep=",",
            skiprows=header_row,
            header=0,
            engine="python",
            on_bad_lines="skip"
        )

        df_check = clean_column_names(df_check)

        normalized_columns = {
            str(column)
            .replace("\ufeff", "")
            .replace("\xa0", " ")
            .strip()
            .lower()
            .replace(" ", "_")
            for column in df_check.columns
        }

        missing_columns = (
            required_columns - normalized_columns
        )

        file_column_check.append(
            {
                "month_folder": file.parent.name,
                "source_file": file.name,
                "detected_header_row": header_row,
                "row_count": len(df_check),
                "column_count": len(df_check.columns),
                "has_account_number": (
                    "account_number" in normalized_columns
                ),
                "has_value_date": (
                    "value_date" in normalized_columns
                ),
                "required_columns_present": (
                    len(missing_columns) == 0
                ),
                "missing_columns": " | ".join(
                    sorted(missing_columns)
                ),
                "columns": " | ".join(
                    map(str, df_check.columns)
                )
            }
        )

    except Exception as error:
        file_column_check.append(
            {
                "month_folder": file.parent.name,
                "source_file": file.name,
                "detected_header_row": np.nan,
                "row_count": np.nan,
                "column_count": np.nan,
                "has_account_number": False,
                "has_value_date": False,
                "required_columns_present": False,
                "missing_columns": "ERROR",
                "columns": f"ERROR: {error}"
            }
        )

column_check_df = pd.DataFrame(file_column_check)

display(column_check_df)

,month_folder,source_file,detected_header_row,row_count,column_count,has_account_number,has_value_date,required_columns_present,missing_columns,columns
0,Apr,UOB.csv,2,44,20,True,True,True,,D1 | Account Number | Value Date | Date | Time...
1,Feb,statment UOB-ก.พ.69.csv,3,40,20,True,True,True,,D1 | Account Number | Value Date | Date | Time...
2,Jan,UOB-6041259598-ม.ค.69.csv,2,30,20,True,True,True,,D1 | Account Number | Value Date | Date | Time...
3,Mar,UOB.csv,3,50,20,True,True,True,,D1 | Account Number | Value Date | Date | Time...
4,May,UOB-พ.ค.69.csv,3,46,20,True,True,True,,D1 | Account Number | Value Date | Date | Time...


In [12]:
def normalize_text(value):
    """Normalize text used for header detection."""
    if value is None:
        return ""

    return (
        str(value)
        .replace("\ufeff", "")
        .replace("\xa0", " ")
        .strip()
        .lower()
    )


def find_uob_header_row(file, max_rows=30):
    """Find the row containing the UOB transaction headers."""
    with open(
        file,
        mode="r",
        encoding="utf-8-sig",
        errors="replace",
        newline=""
    ) as csv_file:

        reader = csv.reader(csv_file)

        for row_index, row in enumerate(reader):
            if row_index >= max_rows:
                break

            row_text = " | ".join(
                normalize_text(value)
                for value in row
            )

            row_text = re.sub(r"\s+", " ", row_text)

            has_account_number = (
                "account number" in row_text
                or "account_number" in row_text
            )

            has_value_date = (
                "value date" in row_text
                or "value_date" in row_text
            )

            if has_account_number and has_value_date:
                return row_index

    return None

In [13]:
def clean_column_names(df):
    df = df.copy()

    df.columns = (
        df.columns
        .astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.replace("\xa0", " ", regex=False)
        .str.replace("\n", " ", regex=False)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
        .str.replace(r"[^a-z0-9_]+", "_", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )

    # Remove unnamed or completely blank columns
    df = df.loc[
        :,
        ~df.columns.str.match(r"^(unnamed|nan|none)?$")
    ]

    return df

In [14]:
all_uob_data = []
file_load_log = []

for file in uob_files:
    try:
        header_row = find_uob_header_row(file)

        if header_row is None:
            raise ValueError(
                "Cannot find Account Number and Value Date header"
            )

        df = pd.read_csv(
            file,
            encoding="utf-8-sig",
            sep=",",
            skiprows=header_row,
            header=0,
            engine="python",
            on_bad_lines="skip",
            dtype="string"
        )

        df = clean_column_names(df)

        # Remove fully empty rows
        df = df.dropna(how="all").copy()

        # Add source information
        df.insert(0, "month_folder", file.parent.name)
        df.insert(1, "source_file", file.name)
        df.insert(2, "source_row_number", range(1, len(df) + 1))

        all_uob_data.append(df)

        file_load_log.append(
            {
                "month_folder": file.parent.name,
                "source_file": file.name,
                "header_row": header_row,
                "rows_loaded": len(df),
                "columns_loaded": len(df.columns),
                "status": "Loaded"
            }
        )

    except Exception as error:
        file_load_log.append(
            {
                "month_folder": file.parent.name,
                "source_file": file.name,
                "header_row": np.nan,
                "rows_loaded": 0,
                "columns_loaded": 0,
                "status": f"ERROR: {error}"
            }
        )

In [15]:
if not all_uob_data:
    raise ValueError("No UOB files were successfully loaded.")

uob_consolidated = pd.concat(
    all_uob_data,
    ignore_index=True,
    sort=False
)

load_log_df = pd.DataFrame(file_load_log)

display(load_log_df)
display(uob_consolidated.head())

,month_folder,source_file,header_row,rows_loaded,columns_loaded,status
0,Apr,UOB.csv,2,43,23,Loaded
1,Feb,statment UOB-ก.พ.69.csv,3,40,24,Loaded
2,Jan,UOB-6041259598-ม.ค.69.csv,2,29,23,Loaded
3,Mar,UOB.csv,3,50,24,Loaded
4,May,UOB-พ.ค.69.csv,3,46,24,Loaded


,month_folder,source_file,source_row_number,d1,account_number,value_date,date,time,description,your_reference,...,reference7,reference8,reference9,reference10,reference11,reference12,deposit,withdrawal,ledger_balance,unnamed_20
0,Apr,UOB.csv,1,D2,6041259598,02/04/2026,02/04/2026,07:00:29 AM,MISC CR,UOB CARD SETTLEMENT,...,,,,,,,"25,130.08",0,"1,863,514.86",<NA>
1,Apr,UOB.csv,2,D2,6041259598,03/04/2026,03/04/2026,07:00:28 AM,MISC CR,UOB CARD SETTLEMENT,...,,,,,,,"57,753.00",0,"1,921,267.86",<NA>
2,Apr,UOB.csv,3,D2,6041259598,03/04/2026,03/04/2026,07:00:28 AM,MISC CR,UOB CARD SETTLEMENT,...,,,,,,,"33,929.89",0,"1,955,197.75",<NA>
3,Apr,UOB.csv,4,D2,6041259598,07/04/2026,07/04/2026,07:00:28 AM,MISC CR,UOB CARD SETTLEMENT,...,,,,,,,"30,262.57",0,"1,985,460.32",<NA>
4,Apr,UOB.csv,5,D2,6041259598,07/04/2026,07/04/2026,07:00:29 AM,MISC CR,UOB CARD SETTLEMENT,...,,,,,,,"19,679.00",0,"2,005,139.32",<NA>


In [16]:
one_row_per_file = (
    uob_consolidated
    .groupby(
        ["month_folder", "source_file"],
        as_index=False,
        dropna=False
    )
    .first()
)

display(one_row_per_file)

,month_folder,source_file,source_row_number,d1,account_number,value_date,date,time,description,your_reference,...,reference7,reference8,reference9,reference10,reference11,reference12,deposit,withdrawal,ledger_balance,unnamed_20
0,Apr,UOB.csv,1,D2,6041259598,02/04/2026,02/04/2026,07:00:29 AM,MISC CR,UOB CARD SETTLEMENT,...,,,,,,,"25,130.08",0,"1,863,514.86",<NA>
1,Feb,statment UOB-ก.พ.69.csv,1,D2,6041259598,02/02/2026,02/02/2026,07:03:04 AM,MISC CR,UOB CARD SETTLEMENT,...,,,,,,,"43,018.29",0.00,"505,675.74",<NA>
2,Jan,UOB-6041259598-ม.ค.69.csv,1,D2,6041259598,05/01/2026,05/01/2026,08:43:58 AM,MISC CR- IFT,9999999999,...,,,,,,,"145,076.23",0,"146,076.43",<NA>
3,Mar,UOB.csv,1,D2,6041259598,04/03/2026,04/03/2026,07:00:35 AM,MISC CR,UOB CARD SETTLEMENT,...,,,,,,,"35,826.11",0.00,"862,486.50",<NA>
4,May,UOB-พ.ค.69.csv,1,D2,6041259598,05/05/2026,05/05/2026,07:06:29 AM,MISC CR,UOB CARD SETTLEMENT,...,,,,,,,"50,042.97",0.00,"1,629,167.74",<NA>


In [17]:
invalid_account_values = {
    "header desc",
    "header2",
    "detail desc",
    "detail",
    "trailer"
}

uob_consolidated = (
    uob_consolidated
    .loc[
        ~uob_consolidated["account_number"]
        .astype("string")
        .str.strip()
        .str.lower()
        .isin(invalid_account_values)
    ]
    .reset_index(drop=True)
)

display(uob_consolidated)

,month_folder,source_file,source_row_number,d1,account_number,value_date,date,time,description,your_reference,...,reference7,reference8,reference9,reference10,reference11,reference12,deposit,withdrawal,ledger_balance,unnamed_20
0,Apr,UOB.csv,1,D2,6041259598,02/04/2026,02/04/2026,07:00:29 AM,MISC CR,UOB CARD SETTLEMENT,...,,,,,,,"25,130.08",0,"1,863,514.86",<NA>
1,Apr,UOB.csv,2,D2,6041259598,03/04/2026,03/04/2026,07:00:28 AM,MISC CR,UOB CARD SETTLEMENT,...,,,,,,,"57,753.00",0,"1,921,267.86",<NA>
2,Apr,UOB.csv,3,D2,6041259598,03/04/2026,03/04/2026,07:00:28 AM,MISC CR,UOB CARD SETTLEMENT,...,,,,,,,"33,929.89",0,"1,955,197.75",<NA>
3,Apr,UOB.csv,4,D2,6041259598,07/04/2026,07/04/2026,07:00:28 AM,MISC CR,UOB CARD SETTLEMENT,...,,,,,,,"30,262.57",0,"1,985,460.32",<NA>
4,Apr,UOB.csv,5,D2,6041259598,07/04/2026,07/04/2026,07:00:29 AM,MISC CR,UOB CARD SETTLEMENT,...,,,,,,,"19,679.00",0,"2,005,139.32",<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
178,May,UOB-พ.ค.69.csv,37,D2,6041259598,26/05/2026,26/05/2026,07:00:39 AM,MISC CR,UOB CARD SETTLEMENT,...,,,,,,,"51,506.80",0.00,"1,109,442.07",<NA>
179,May,UOB-พ.ค.69.csv,38,D2,6041259598,26/05/2026,26/05/2026,07:00:39 AM,MISC CR,UOB CARD SETTLEMENT,...,,,,,,,"29,829.42",0.00,"1,139,271.49",<NA>
180,May,UOB-พ.ค.69.csv,39,D2,6041259598,28/05/2026,28/05/2026,07:00:27 AM,MISC CR,UOB CARD SETTLEMENT,...,,,,,,,"40,427.10",0.00,"1,179,698.59",<NA>
181,May,UOB-พ.ค.69.csv,40,D2,6041259598,28/05/2026,28/05/2026,07:00:27 AM,MISC CR,UOB CARD SETTLEMENT,...,,,,,,,"39,811.07",0.00,"1,219,509.66",<NA>


In [18]:
# Add bank name
uob_consolidated["bank"] = "UOB"

# Columns to keep
columns_to_keep = [
    "bank",
    "account_number",
    "date",
    "time",
    "description",
    "your_reference",
    "deposit",
    "withdrawal",
    "ledger_balance"
]

# Keep only columns that actually exist
existing_columns = [
    column
    for column in columns_to_keep
    if column in uob_consolidated.columns
]

uob_consolidated = (
    uob_consolidated[existing_columns]
    .copy()
)

display(uob_consolidated)

,bank,account_number,date,time,description,your_reference,deposit,withdrawal,ledger_balance
0,UOB,6041259598,02/04/2026,07:00:29 AM,MISC CR,UOB CARD SETTLEMENT,"25,130.08",0,"1,863,514.86"
1,UOB,6041259598,03/04/2026,07:00:28 AM,MISC CR,UOB CARD SETTLEMENT,"57,753.00",0,"1,921,267.86"
2,UOB,6041259598,03/04/2026,07:00:28 AM,MISC CR,UOB CARD SETTLEMENT,"33,929.89",0,"1,955,197.75"
3,UOB,6041259598,07/04/2026,07:00:28 AM,MISC CR,UOB CARD SETTLEMENT,"30,262.57",0,"1,985,460.32"
4,UOB,6041259598,07/04/2026,07:00:29 AM,MISC CR,UOB CARD SETTLEMENT,"19,679.00",0,"2,005,139.32"
...,...,...,...,...,...,...,...,...,...
178,UOB,6041259598,26/05/2026,07:00:39 AM,MISC CR,UOB CARD SETTLEMENT,"51,506.80",0.00,"1,109,442.07"
179,UOB,6041259598,26/05/2026,07:00:39 AM,MISC CR,UOB CARD SETTLEMENT,"29,829.42",0.00,"1,139,271.49"
180,UOB,6041259598,28/05/2026,07:00:27 AM,MISC CR,UOB CARD SETTLEMENT,"40,427.10",0.00,"1,179,698.59"
181,UOB,6041259598,28/05/2026,07:00:27 AM,MISC CR,UOB CARD SETTLEMENT,"39,811.07",0.00,"1,219,509.66"


In [19]:
uob_consolidated = (
    uob_consolidated
    .dropna(subset=["account_number"])
    .reset_index(drop=True)
)

display(uob_consolidated)

,bank,account_number,date,time,description,your_reference,deposit,withdrawal,ledger_balance
0,UOB,6041259598,02/04/2026,07:00:29 AM,MISC CR,UOB CARD SETTLEMENT,"25,130.08",0,"1,863,514.86"
1,UOB,6041259598,03/04/2026,07:00:28 AM,MISC CR,UOB CARD SETTLEMENT,"57,753.00",0,"1,921,267.86"
2,UOB,6041259598,03/04/2026,07:00:28 AM,MISC CR,UOB CARD SETTLEMENT,"33,929.89",0,"1,955,197.75"
3,UOB,6041259598,07/04/2026,07:00:28 AM,MISC CR,UOB CARD SETTLEMENT,"30,262.57",0,"1,985,460.32"
4,UOB,6041259598,07/04/2026,07:00:29 AM,MISC CR,UOB CARD SETTLEMENT,"19,679.00",0,"2,005,139.32"
...,...,...,...,...,...,...,...,...,...
176,UOB,6041259598,26/05/2026,07:00:39 AM,MISC CR,UOB CARD SETTLEMENT,"51,506.80",0.00,"1,109,442.07"
177,UOB,6041259598,26/05/2026,07:00:39 AM,MISC CR,UOB CARD SETTLEMENT,"29,829.42",0.00,"1,139,271.49"
178,UOB,6041259598,28/05/2026,07:00:27 AM,MISC CR,UOB CARD SETTLEMENT,"40,427.10",0.00,"1,179,698.59"
179,UOB,6041259598,28/05/2026,07:00:27 AM,MISC CR,UOB CARD SETTLEMENT,"39,811.07",0.00,"1,219,509.66"


In [20]:
# Check column data types
print(uob_consolidated.dtypes)

# Check sample values
display(uob_consolidated.head())

# Check missing values
display(
    uob_consolidated
    .isna()
    .sum()
    .rename("missing_rows")
    .to_frame()
)

bank                 str
account_number    string
date              string
time              string
description       string
your_reference    string
deposit           string
withdrawal        string
ledger_balance    string
dtype: object


,bank,account_number,date,time,description,your_reference,deposit,withdrawal,ledger_balance
0,UOB,6041259598,02/04/2026,07:00:29 AM,MISC CR,UOB CARD SETTLEMENT,"25,130.08",0,"1,863,514.86"
1,UOB,6041259598,03/04/2026,07:00:28 AM,MISC CR,UOB CARD SETTLEMENT,"57,753.00",0,"1,921,267.86"
2,UOB,6041259598,03/04/2026,07:00:28 AM,MISC CR,UOB CARD SETTLEMENT,"33,929.89",0,"1,955,197.75"
3,UOB,6041259598,07/04/2026,07:00:28 AM,MISC CR,UOB CARD SETTLEMENT,"30,262.57",0,"1,985,460.32"
4,UOB,6041259598,07/04/2026,07:00:29 AM,MISC CR,UOB CARD SETTLEMENT,"19,679.00",0,"2,005,139.32"


,missing_rows
bank,0
account_number,0
date,0
time,0
description,0
your_reference,0
deposit,0
withdrawal,0
ledger_balance,0


In [21]:
uob_consolidated = uob_consolidated.copy()

# Clean original date/time text
uob_consolidated["date"] = (
    uob_consolidated["date"]
    .astype("string")
    .str.strip()
)

uob_consolidated["time"] = (
    uob_consolidated["time"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Convert date: source is DD/MM/YYYY
uob_consolidated["date"] = pd.to_datetime(
    uob_consolidated["date"],
    dayfirst=True,
    errors="coerce"
)

# Convert time to hour 0–23
uob_consolidated["time"] = (
    pd.to_datetime(
        uob_consolidated["time"].astype("string").str.strip(),
        format="mixed",
        errors="coerce"
    )
    .dt.strftime("%H:%M")
    .astype("string")
)

# Financial columns
money_columns = [
    "deposit",
    "withdrawal",
    "ledger_balance"
]

for column in money_columns:
    uob_consolidated[column] = (
        uob_consolidated[column]
        .astype("string")
        .str.replace(",", "", regex=False)
        .str.strip()
        .pipe(pd.to_numeric, errors="coerce")
        .round()
        .astype("Int64")
    )

# Remove rows without account number
uob_consolidated["account_number"] = (
    uob_consolidated["account_number"]
    .astype("string")
    .str.strip()
)

uob_consolidated = (
    uob_consolidated
    .dropna(subset=["account_number"])
    .loc[lambda df: df["account_number"].ne("")]
    .reset_index(drop=True)
)

display(uob_consolidated)
print(uob_consolidated.dtypes)

,bank,account_number,date,time,description,your_reference,deposit,withdrawal,ledger_balance
0,UOB,6041259598,2026-04-02,07:00,MISC CR,UOB CARD SETTLEMENT,25130,0,1863515
1,UOB,6041259598,2026-04-03,07:00,MISC CR,UOB CARD SETTLEMENT,57753,0,1921268
2,UOB,6041259598,2026-04-03,07:00,MISC CR,UOB CARD SETTLEMENT,33930,0,1955198
3,UOB,6041259598,2026-04-07,07:00,MISC CR,UOB CARD SETTLEMENT,30263,0,1985460
4,UOB,6041259598,2026-04-07,07:00,MISC CR,UOB CARD SETTLEMENT,19679,0,2005139
...,...,...,...,...,...,...,...,...,...
176,UOB,6041259598,2026-05-26,07:00,MISC CR,UOB CARD SETTLEMENT,51507,0,1109442
177,UOB,6041259598,2026-05-26,07:00,MISC CR,UOB CARD SETTLEMENT,29829,0,1139271
178,UOB,6041259598,2026-05-28,07:00,MISC CR,UOB CARD SETTLEMENT,40427,0,1179699
179,UOB,6041259598,2026-05-28,07:00,MISC CR,UOB CARD SETTLEMENT,39811,0,1219510


bank                         str
account_number            string
date              datetime64[us]
time                      string
description               string
your_reference            string
deposit                    Int64
withdrawal                 Int64
ledger_balance             Int64
dtype: object


In [22]:
uob_df = uob_consolidated.copy()

# Clean column names
uob_df.columns = (
    uob_df.columns
    .astype(str)
    .str.strip()
    .str.lower()
)

# Rename UOB columns into KBANK format
uob_df = uob_df.rename(
    columns={
        "date": "transaction_date",
        "time": "transaction_time",
        "ledger_balance": "outstanding_balance",
        "description": "transaction_type",
        "your_reference": "description"
    }
)

# Convert date
uob_df["transaction_date"] = pd.to_datetime(
    uob_df["transaction_date"],
    errors="coerce"
)

# Clean transaction time
uob_df["transaction_time"] = (
    uob_df["transaction_time"]
    .astype("string")
    .str.strip()
    .replace(["", "nan", "NaN", "<NA>"], pd.NA)
)

# Convert numeric columns
for column in ["withdrawal", "deposit", "outstanding_balance"]:
    uob_df[column] = pd.to_numeric(
        uob_df[column],
        errors="coerce"
    )

# Replace zero withdrawal/deposit with missing values
uob_df["withdrawal"] = uob_df["withdrawal"].mask(
    uob_df["withdrawal"].eq(0)
)

uob_df["deposit"] = uob_df["deposit"].mask(
    uob_df["deposit"].eq(0)
)

# Keep KBANK standardized format
col = [
    "bank",
    "account_number",
    "transaction_date",
    "transaction_time",
    "withdrawal",
    "deposit",
    "outstanding_balance",
    "transaction_type",
    "description"
]

uob_df = uob_df[col]

display(uob_df.head())
print(uob_df.dtypes)

,bank,account_number,transaction_date,transaction_time,withdrawal,deposit,outstanding_balance,transaction_type,description
0,UOB,6041259598,2026-04-02,07:00,<NA>,25130,1863515,MISC CR,UOB CARD SETTLEMENT
1,UOB,6041259598,2026-04-03,07:00,<NA>,57753,1921268,MISC CR,UOB CARD SETTLEMENT
2,UOB,6041259598,2026-04-03,07:00,<NA>,33930,1955198,MISC CR,UOB CARD SETTLEMENT
3,UOB,6041259598,2026-04-07,07:00,<NA>,30263,1985460,MISC CR,UOB CARD SETTLEMENT
4,UOB,6041259598,2026-04-07,07:00,<NA>,19679,2005139,MISC CR,UOB CARD SETTLEMENT


bank                              str
account_number                 string
transaction_date       datetime64[us]
transaction_time               string
withdrawal                      Int64
deposit                         Int64
outstanding_balance             Int64
transaction_type               string
description                    string
dtype: object


In [23]:
import sqlite3
from pathlib import Path
import pandas as pd

# Convert transaction date
uob_df["transaction_date"] = pd.to_datetime(
    uob_df["transaction_date"],
    errors="coerce"
)

export_df = uob_df.dropna(subset=["transaction_date"]).copy()

# Current notebook is inside Python_Command
project_root = Path.cwd().parent
raw_folder = project_root / "Datamart" / "Raw"
raw_folder.mkdir(parents=True, exist_ok=True)

for year, year_df in export_df.groupby(
    export_df["transaction_date"].dt.year
):
    year = int(year)

    database_path = raw_folder / f"uob_{year}.db"
    table_name = f"uob_{year}"

    with sqlite3.connect(database_path) as connection:
        year_df.to_sql(
            name=table_name,
            con=connection,
            if_exists="replace",
            index=False,
            chunksize=10_000
        )

    print(f"Exported {len(year_df):,} rows")
    print(f"Database: {database_path.resolve()}")
    print(f"Table: {table_name}")

Exported 181 rows
Database: C:\Users\User\Desktop\Python 100 Days\Bank_Account_Project\Datamart\Raw\uob_2026.db
Table: uob_2026
